# Can we predict next-month account churn?
Target: `churn_next_month`. Data: monthly account snapshots (`data.csv`), dictionary in `columns.csv`.

Plan: inspect data, check for leakage, build features available at snapshot time, validate by time, report honest performance.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(df.shape)
print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

In [ ]:
print(df.isna().sum().sum(), 'missing;', df.duplicated(['account_ref','snapshot_month']).sum(), 'duplicate account-months;', df.account_ref.nunique(), 'accounts')
print('churn rate:', df.churn_next_month.mean().round(3))
df.describe().T

## Churn over time
Churn is strongly non-stationary: it spikes to 40-55% for Nov 2024 to Jan 2025. Validation must be by time, not random.

In [ ]:
m=df.groupby('snapshot_month').churn_next_month.agg(['mean','size'])
ax=m['mean'].plot(marker='o',figsize=(9,3)); ax.set_ylabel('churn rate'); ax.set_title('Churn rate by snapshot month'); plt.xticks(rotation=60); plt.show()
print(df.groupby('plan').churn_next_month.mean().round(3))

## Leakage check: `lifetime_months`
The dictionary says it is the *total months the account was active, from a CRM export on 2026-01-05*, so it is only known after the account's life has ended and would not exist at prediction time. Compare it with `tenure_months`:

In [ ]:
df['gap']=df.lifetime_months-df.tenure_months
print(df.groupby('churn_next_month').gap.describe())
print('P(churn | lifetime==tenure) =', df.loc[df.gap==0,'churn_next_month'].mean().round(3),
      '| P(churn | lifetime>tenure) =', df.loc[df.gap>0,'churn_next_month'].mean())

Every churning row has `lifetime_months == tenure_months`, and no non-churning row does. The column encodes the label directly. **It is excluded from all real models**; a deliberately leaky model is shown below only so the inflated score isn't mistaken for skill.

The remaining features (`plan, seats, monthly_fee, tenure_months, usage_hours, support_tickets`) are measured in the snapshot month. I add per-account history features built only from *earlier* snapshots, plus calendar month.

In [ ]:
df=df.sort_values(['account_ref','snapshot_month']).reset_index(drop=True)
df['period']=pd.PeriodIndex(df.snapshot_month,freq='M')
g=df.groupby('account_ref')
df['usage_delta']=df.usage_hours-g.usage_hours.shift(1)
df['usage_vs_avg']=df.usage_hours/g.usage_hours.transform(lambda s:s.shift(1).expanding().mean())
df['tickets_prev3']=g.support_tickets.transform(lambda s:s.shift(1).rolling(3,min_periods=1).sum())
df['fee_per_seat']=df.monthly_fee/df.seats
df['cal_month']=df.period.dt.month
df=pd.concat([df,pd.get_dummies(df.plan,prefix='plan',dtype=float)],axis=1)
base=['seats','monthly_fee','tenure_months','usage_hours','support_tickets','fee_per_seat','plan_basic','plan_pro','plan_team']
hist=['usage_delta','usage_vs_avg','tickets_prev3']
cal=['cal_month']
y='churn_next_month'

## Validation
Train on snapshots up to 2025-05, test on 2025-06 to 2025-11 (later, unseen months). Test accounts may also appear in training with earlier months, which matches deployment (scoring existing accounts). The training period contains only one winter spike, so a calendar-month feature has a single year to learn from; I test with and without it.

In [ ]:
tr=df[df.period<='2025-05']; te=df[df.period>'2025-05']
print(len(tr),len(te),'base rate train/test:',tr[y].mean().round(3),te[y].mean().round(3))
def fit_eval(name,feats,model):
    fill=-1 if model=='gbm' else 0
    m=(HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0) if model=='gbm'
       else make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)))
    m.fit(tr[feats].fillna(fill),tr[y]); p=m.predict_proba(te[feats].fillna(fill))[:,1]
    return dict(model=name,AUC=roc_auc_score(te[y],p),AP=average_precision_score(te[y],p)),m,p
rows=[]; preds={}
for name,feats,mt in [('Logistic: base',base,'lr'),('GBM: base',base,'gbm'),
                      ('GBM: base+history',base+hist,'gbm'),('GBM: base+history+calendar',base+hist+cal,'gbm'),
                      ('LEAKY GBM: + lifetime_months (do not use)',base+hist+['lifetime_months'],'gbm')]:
    r,m,p=fit_eval(name,feats,mt); rows.append(r); preds[name]=(m,p)
res=pd.DataFrame(rows).set_index('model'); res.loc['Random / base rate']=[0.5,te[y].mean()]
res.round(3)

AUC 0.5 and AP equal to the base rate are chance level. The leaky row shows the near-perfect score you would get by mistakenly keeping `lifetime_months`.

In [ ]:
from sklearn.inspection import permutation_importance
f=base+hist; m,_=preds['GBM: base+history']
pi=permutation_importance(m,te[f].fillna(-1),te[y],scoring='roc_auc',n_repeats=5,random_state=0)
pd.Series(pi.importances_mean,index=f).sort_values().plot.barh(figsize=(6,3.5),title='Permutation importance (AUC drop)'); plt.show()

In [ ]:
best=max([k for k in preds if 'LEAKY' not in k],key=lambda k:roc_auc_score(te[y],preds[k][1])); print('best honest model:',best)
t=te[[y]].assign(p=preds[best][1]); t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=False)+1
d=t.groupby('decile').agg(pred=('p','mean'),actual=(y,'mean'),n=(y,'size')).round(3); d['lift']=(d.actual/te[y].mean()).round(2); d.sort_index(ascending=False)

## Conclusions (from the run on this data)
- **Yes, but only modestly.** Honest models reach test AUC about 0.68-0.70 (chance 0.5), average precision about 0.22 vs a 0.093 base rate. The top-decile of scored accounts churns at about 2.8x the average rate.
- **`lifetime_months` is leakage** (equals `tenure_months` exactly when the account churns). With it AUC is about 0.998, which is not real. Exclude it.
- Plan, seats and fee carry no churn signal on their own; the useful signal is weak and sits in tenure, usage and tickets.
- History features and calendar month did not help (slightly worse out of sample). With only one winter in the data, seasonality cannot be learned reliably.
- **Drift:** churn was 18% in training and 9% in the test period, so predicted probabilities are too high in absolute terms (decile 10 predicted 0.42, actual 0.26). Use scores for ranking, or recalibrate on recent data, before using them as probabilities.
- Next steps: more history (2+ years) to model seasonality, cohort/reason-for-cancel data, and recalibration.